## M4 — Clustering K-Means
Segmentamos las 100 ubicaciones en perfiles para enriquecer la narrativa ante el jurado.

In [28]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

RUTA_BASE = r'C:\Users\jcrs2\Documents\Akademia\akademia-residencias'

df = pd.read_csv(rf'{RUTA_BASE}\outputs\top100_ubicaciones.csv', encoding='utf-8-sig')

print(f"Ubicaciones cargadas: {df.shape[0]}")
print(f"Columnas: {df.columns.tolist()}")

Ubicaciones cargadas: 50
Columnas: ['CPRO', 'CMUN', 'NOMBRE', 'secciones', 'fofm_medio', 'fofm_max', 'vol_75plus_total', 'vol_80plus_total', 'pct_75plus_media', 'ratio_viudez_media', 'cohorte_futura', 'tasa_sustitucion', 'renta_hogar', 'pension_media', 'precio_m2', 'indice_cobertura', 'tiene_hospital', 'latitud', 'longitud', 'ranking_municipio', 'posicion_final']


### Paso 1 — Seleccionar variables y escalar

In [29]:
# Variables que definen el perfil de cada municipio
features = [
    'renta_hogar',
    'vol_75plus_total',
    'pct_75plus_media',
    'ratio_viudez_media',
    'cohorte_futura'
]

X = df[features].copy()

# Escalar — media 0, desviación estándar 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Variables escaladas:")
print(pd.DataFrame(X_scaled, columns=features).describe().round(2))

Variables escaladas:
       renta_hogar  vol_75plus_total  pct_75plus_media  ratio_viudez_media  \
count        50.00             50.00             50.00               50.00   
mean         -0.00             -0.00              0.00               -0.00   
std           1.01              1.01              1.01                1.01   
min          -0.62             -0.39             -1.96               -1.67   
25%          -0.52             -0.37             -0.68               -0.50   
50%          -0.28             -0.32             -0.01               -0.10   
75%          -0.02             -0.15              0.82                0.50   
max           4.84              6.19              1.66                4.05   

       cohorte_futura  
count           50.00  
mean            -0.00  
std              1.01  
min             -0.42  
25%             -0.39  
50%             -0.33  
75%             -0.12  
max              6.06  


### Paso 2 — Encontrar el número óptimo de clusters

In [30]:
resultados = []

for k in range(2, 9):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    sil    = silhouette_score(X_scaled, labels)
    inercia = kmeans.inertia_
    resultados.append({'k': k, 'silhouette': round(sil, 4), 'inercia': round(inercia, 1)})

df_res = pd.DataFrame(resultados)
print(df_res.to_string(index=False))

 k  silhouette  inercia
 2      0.7172    172.5
 3      0.2880    121.2
 4      0.3790     85.6
 5      0.3101     63.7
 6      0.3432     52.0
 7      0.3472     41.5
 8      0.3530     34.0


### Paso 3 — Aplicar K-Means con 4 clusters

In [31]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Ver cuántos municipios hay en cada cluster
print("Municipios por cluster:")
print(df['cluster'].value_counts().sort_index())

# Ver las características medias de cada cluster
perfil = df.groupby('cluster')[features].mean().round(0)
print("\nPerfil medio por cluster:")
print(perfil.to_string())

Municipios por cluster:
cluster
0    33
1     1
2    12
3     4
Name: count, dtype: int64

Perfil medio por cluster:
         renta_hogar  vol_75plus_total  pct_75plus_media  ratio_viudez_media  cohorte_futura
cluster                                                                                     
0            34771.0            3006.0               9.0                 2.0          3733.0
1            37371.0          186908.0              12.0                 2.0        171910.0
2            35520.0           24541.0              12.0                 2.0         25078.0
3            66070.0            4997.0               8.0                 2.0          6564.0


### Paso 4 — Etiquetar los clusters con nombres estratégicos

In [32]:
nombres_cluster = {
    0: "Zona envejecida — solvencia ajustada",
    1: "Ciudad media — mercado relevante",
    2: "Zona premium — máxima solvencia",
    3: "Gran ciudad — mercado masivo"
}

df['perfil'] = df['cluster'].map(nombres_cluster)

resumen = df.groupby('perfil').agg(
    municipios      = ('NOMBRE', 'count'),
    renta_media     = ('renta_hogar', 'mean'),
    vol_75plus      = ('vol_75plus_total', 'mean'),
    fofm_medio      = ('fofm_medio', 'mean')
).round(2)

print("=== PERFILES ESTRATÉGICOS ACTUALIZADOS ===")
print(resumen.to_string())

for perfil in nombres_cluster.values():
    municipios = df[df['perfil'] == perfil]['NOMBRE'].tolist()
    print(f"\n{perfil} ({len(municipios)}):")
    print(", ".join(municipios[:8]))
    if len(municipios) > 8:
        print(f"  ... y {len(municipios)-8} más")

=== PERFILES ESTRATÉGICOS ACTUALIZADOS ===
                                      municipios  renta_media  vol_75plus  fofm_medio
perfil                                                                               
Ciudad media — mercado relevante               1     37371.00   186908.00        0.44
Gran ciudad — mercado masivo                   4     66069.75     4997.25        0.51
Zona envejecida — solvencia ajustada          33     34771.06     3005.82        0.26
Zona premium — máxima solvencia               12     35519.50    24541.33        0.30

Zona envejecida — solvencia ajustada (33):
Esplugues de Llobregat, Galdakao, Hondarribia, Eivissa, Godella, Bormujos, Eliana, l', Maó-Mahón
  ... y 25 más

Ciudad media — mercado relevante (1):
Barcelona

Zona premium — máxima solvencia (12):
Bilbao, Donostia/San Sebastián, Tolosa, Pamplona/Iruña, Getxo, Vitoria-Gasteiz, Santander, Zaragoza
  ... y 4 más

Gran ciudad — mercado masivo (4):
Pozuelo de Alarcón, Majadahonda, Torrelodones, S

### Paso 5 — Guardar resultados del clustering

In [33]:
df.to_csv(rf'{RUTA_BASE}\outputs\top100_con_clusters.csv', index=False, encoding='utf-8-sig')

df_muni      = pd.read_csv(rf'{RUTA_BASE}\outputs\ranking_municipios.csv', encoding='utf-8-sig')
resumen_prov = pd.read_csv(rf'{RUTA_BASE}\outputs\resumen_provincias.csv', encoding='utf-8-sig')

with pd.ExcelWriter(rf'{RUTA_BASE}\outputs\resultados_akademia.xlsx', engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Top 100 ubicaciones', index=False)
    df_muni.head(500).to_excel(writer, sheet_name='Ranking municipios', index=False)
    resumen_prov.to_excel(writer, sheet_name='Resumen por provincia', index=False)
    
    # Nueva pestaña con resumen de perfiles
    resumen_clusters = df.groupby('perfil').agg(
        municipios  = ('NOMBRE', 'count'),
        renta_media = ('renta_hogar', 'mean'),
        vol_75plus  = ('vol_75plus_total', 'mean'),
        fofm_medio  = ('fofm_medio', 'mean')
    ).round(2).reset_index()
    resumen_clusters.to_excel(writer, sheet_name='Perfiles estratégicos', index=False)

print("Archivos guardados:")
print(f"  top100_con_clusters.csv → {len(df)} municipios con perfil asignado")
print(f"  resultados_akademia.xlsx → actualizado con pestaña de perfiles")

Archivos guardados:
  top100_con_clusters.csv → 50 municipios con perfil asignado
  resultados_akademia.xlsx → actualizado con pestaña de perfiles
